In [ ]:
# --- Q1: Count works with "artificial intelligence" in the title ---
import requests

BASE_URL = "exec(open("./scripts/import_packages.py").read())"

# title.search limits matching to titles only;
# double quotes make it an exact phrase match
params = {"filter": 'title.search:"artificial intelligence"'}

response = requests.get(BASE_URL, params=params)
response.raise_for_status()   # quality check: stop if the call failed

data = response.json()        # API returns JSON, parsed into a dict
ai_title_count = data["meta"]["count"]

print(f"Works with 'artificial intelligence' in the title: {ai_title_count:,}")

Works with 'artificial intelligence' in the title: 330,744


In [4]:
q1_params = {"filter": 'title.search:"artificial intelligence"'}

q1_response = requests.get(BASE_URL, params=q1_params)
q1_response.raise_for_status()   # quality check: stop if the call failed

q1_data = q1_response.json()     # parse JSON into a Python dictionary

# meta.count = total matching works (not just the first page of 25)
ai_title_count = q1_data["meta"]["count"]

print(f"Works with 'artificial intelligence' in the title: {ai_title_count:,}")

Works with 'artificial intelligence' in the title: 330,744


In [6]:
# --- Q1: Count works with "artificial intelligence" in the title ---
import requests

BASE_URL = "https://api.openalex.org/works"

# title.search limits matching to titles only;
# double quotes make it an exact phrase match
params = {"filter": 'title.search:"artificial intelligence"'}

response = requests.get(BASE_URL, params=params)
response.raise_for_status()   # quality check: stop if the call failed

data = response.json()        # API returns JSON, parsed into a dict
ai_title_count = data["meta"]["count"]

print(f"Works with 'artificial intelligence' in the title: {ai_title_count:,}")

Works with 'artificial intelligence' in the title: 330,744


In [16]:
import pandas as pd
q2_params = {
    # AI in title AND published 2000 or later
    "filter": 'title.search:"artificial intelligence",publication_year:>1999',
    # count works per country of the authors' institutions
    "group_by": "authorships.countries",}

q2_response = requests.get(BASE_URL, params=q2_params)
q2_response.raise_for_status()
q2_data = q2_response.json()

country_df = pd.DataFrame(q2_data["group_by"])

country_df = (
    country_df
    .rename(columns={"key_display_name": "Country",
                     "count": "AI Publications","title": "Article Title"})
    [["Country", "AI Publications", "Article Title" ]]
    .sort_values("AI Publications", ascending=False)
    .reset_index(drop=True)
)
country_df.index = country_df.index + 1   # rank starts at 1

country_df.index.name = "Rank"

display(
    country_df.head(10).style
    .format({"AI Publications": "{:,.0f}"})
    .set_caption("AI publications by country, 2000 onward")
)

KeyError: "['Article Title'] not in index"

In [ ]:
title_params = {
    "filter": 'title.search:"artificial intelligence",publication_year:>1999',
    "sort": "publication_date:desc",
    "per-page": 10
}
title_response = requests.get(BASE_URL, params=title_params)
title_response.raise_for_status()
title_data = title_response.json()

title_df = (
    pd.DataFrame(title_data["results"])[["publication_year", "title"]]
    .rename(columns={
        "publication_year": "Year",
        "title": "Article Title"
    })
    .sort_values("Year", ascending=False)
    .reset_index(drop=True)
)

display(title_df)



,Year,Article Title
0,2045,Artificial Intelligence in Plant Sciences
1,2045,Artificial Intelligence in Plant Sciences
2,2029,Development of an Advanced Artificial Intellig...
3,2028,ARTIFICIAL INTELLIGENCE IN ENGLISH LANGUAGE TE...
4,2028,Combating foreign bribery and corruption : an ...
5,2027,"Entrepreneurial subjectivity, expertise, and s..."
6,2027,[Call for Papers] Ethical implications of arti...
7,2026,ETHICAL CONSIDERATIONS AND POLICY FRAMEWORKS F...
8,2026,ETHICAL CONSIDERATIONS AND POLICY FRAMEWORKS F...
9,2026,TRANSFORMING GOVERNMENT SERVICE DELIVERY THROU...


In [2]:
import requests
import pandas as pd

BASE_URL = "https://api.openalex.org/works"
titles_citations_params = {
    "filter": 'title.search:"artificial intelligence",publication_year:>1999',
    "sort": "cited_by_count:desc",
    "per-page": 10
}

titles_citations_response = requests.get(BASE_URL, params=titles_citations_params)
titles_citations_response.raise_for_status()
titles_citations_data = titles_citations_response.json()

def format_authors_and_institutions(authorships):
    formatted_authors = []
    for authorship in authorships:
        author = authorship.get("author") or {}
        author_name = author.get("display_name", "Unknown author")
        institutions = [
            institution["display_name"]
            for institution in authorship.get("institutions", [])
            if institution.get("display_name")
        ]
        institution_names = "; ".join(institutions) or "Institution not listed"
        formatted_authors.append(f"{author_name} ({institution_names})")
    return "; ".join(formatted_authors)

top_cited_works = titles_citations_data["results"]
top_cited_df = pd.DataFrame([
    {
        "Year": work.get("publication_year"),
        "Paper Title": work.get("title"),
        "Citations": work.get("cited_by_count", 0),
        "Authors and Institutions": format_authors_and_institutions(
            work.get("authorships", [])
        ),
    }
    for work in top_cited_works
])

display(
    top_cited_df.style
    .format({"Citations": "{:,.0f}"})
    .set_properties(
        subset=["Paper Title", "Authors and Institutions"],
        **{"white-space": "normal", "text-align": "left", "max-width": "450px"}
    )
    .set_caption("Top 10 most-cited AI-title papers published since 2000")
)

,Year,Paper Title,Citations,Authors and Institutions
0,2018,High-performance medicine: the convergence of human and artificial intelligence,"10,188",Eric J. Topol (Scripps Research Institute)
1,2019,"Explainable Artificial Intelligence (XAI): Concepts, taxonomies, opportunities and challenges toward responsible AI","10,067",Alejandro Barredo Arrieta (Tecnalia); Natalia Díaz-Rodríguez (Institut national de recherche en sciences et technologies du numérique; École Nationale Supérieure de Techniques Avancées; École Nationale Supérieure de Techniques Avancées Paris; Unité d'Informatique et d'Ingénierie des Systèmes); Javier Del Ser (Tecnalia); Adrien Bennetot (Institut national de recherche en sciences et technologies du numérique; École Nationale Supérieure de Techniques Avancées; Sorbonne Université; Institut Systèmes Intelligents et de Robotique; Unité d'Informatique et d'Ingénierie des Systèmes); Siham Tabik (Universidad de Granada); Alberto Barbado (Telefónica (Spain); Telefonica Research and Development); Salvador García (Universidad de Granada); Sergio Gil-López (Tecnalia); Daniel Molina (Universidad de Granada); Richard Benjamins (Telefónica (Spain); Telefonica Research and Development); Raja Chatila (Sorbonne Université; Institut Systèmes Intelligents et de Robotique); Francisco Herrera (Universidad de Granada)
2,2019,Systematic review of research on artificial intelligence applications in higher education – where are the educators?,"6,754",Olaf Zawacki‐Richter (Carl von Ossietzky Universität Oldenburg); Victoria I. Marín (Carl von Ossietzky Universität Oldenburg); Melissa Bond (Carl von Ossietzky Universität Oldenburg); Franziska Gouverneur (Carl von Ossietzky Universität Oldenburg)
3,2018,Peeking Inside the Black-Box: A Survey on Explainable Artificial Intelligence (XAI),"6,281",Amina Adadi (Sidi Mohamed Ben Abdellah University); Mohammed Berrada (Sidi Mohamed Ben Abdellah University)
4,2005,Proceedings of the 19th International Joint Conference on Artificial Intelligence,"5,782",Josep M. Pujol (Institution not listed); Jordi Delgado (Institution not listed); Ramón Sangüesa (Institution not listed); Andreas Flache (Institution not listed)
5,2018,Explanation in artificial intelligence: Insights from the social sciences,"5,241",Tim Miller (The University of Melbourne)
6,2017,"Artificial intelligence in healthcare: past, present and future","4,885",Fei Jiang (University of Hong Kong); Yong Jiang (Capital Medical University; Beijing Tian Tan Hospital); Hui Zhi (University of Hong Kong); Yi Dong (Fudan University; Huashan Hospital); Hao Li (Institution not listed); Sufeng Ma (Institution not listed); Yilong Wang (Beijing Tian Tan Hospital); Yilong Wang (Fudan University; Beijing Tian Tan Hospital; Huashan Hospital); Qiang Dong (Fudan University; Huashan Hospital; University of Hong Kong); Haipeng Shen (Beijing Tian Tan Hospital; University of Hong Kong); Yongjun Wang (Beijing Tian Tan Hospital); Yongjun Wang (Beijing Tian Tan Hospital)
7,2019,"Artificial Intelligence (AI): Multidisciplinary perspectives on emerging challenges, opportunities, and agenda for research, practice and policy","4,497",Yogesh Kumar Dwivedi (Prin. L. N. Welingkar Institute of Management Development and Research; Swansea University); Laurie Hughes (Swansea University); Elvira Ismagilova (University of Bradford); Gert Aarts (Swansea University); Crispin R. Coombs (Loughborough University); Tom Crick (Swansea University); Yanqing Duan (University of Bedfordshire); Rohita Dwivedi (Prin. L. N. Welingkar Institute of Management Development and Research); John Steven Edwards (Aston University); Aled Eirug (Swansea University); Vassilis Galanos (University of Edinburgh); P. Vigneswara Ilavarasan (Indian Institute of Technology Delhi); Marijn Janssen (Delft University of Technology); Paul Jones (Swansea University); Arpan Kumar Kar (Indian Institute of Technology Delhi); Hatice Kizgin (University of Bradford); Bianca Kronemann (Swansea University); Banita Lal